# Datamine SURVIN Process Example

This notebook demonstrates how to configure and run the **`survin`** process wrapper in `dmstudio`.

## Process Description

## SURVIN

**SURVIN** is a data format conversion process.

**SURVIN** requires as input a file of perimeters and/or strings of points that may represent boundaries of planned or actual mine development. The file may have been output from a mine planning process, or digitized from mine plans.

**SURVIN** creates a file containing a record for each unique point within the perimeter file, with associated point attributes, and a file containing the links between points in the same string and the string identification values. The point and segment files may then be used as input to the survey editor to enable interaction of planned development with survey stations and surveyed features, thus allowing listings to be produced of the measurements required by the surveyor to set out excavation lines for operational production control.

If two points in the perimeter file are within the tolerance distance parameter @**TOL** supplied by the user, it is not duplicated in the output point file but the link is stored in the string segment file for the string concerned. The distance between points is computed between their three-dimensional coordinate positions.

If the input perimeter file does not contain the fields **PSYMBOL** , **PSYMSZE** , **P** , **PTYPE** , **PCODE** , the corresponding parameters must be supplied by the user, and will be assigned as constants in the output point and string files.

### Input Files:

* **perimin** (String):
  Input perimeter file. Must contain the fields XP, YP, ZP, PTN and PVALUE (numeric,
  explicit). Optional fields PTYPE, PCODE, P, PSYMBOL, and PSYMSZE will be used, if
  present. Additional fields found will be added to the output point and string files.
  Required=Yes

### Output Files:

* **pointou** (Undefined):
  Output point file. This will contain fields PID, X, Y, Z, PSYMBOL, PSYMSZE, P and
  PERIOD (numeric, explicit).
  Required=Yes

* **segou** (Undefined):
  Output string segment file. This will contain the fields PID1, PID2, PVALUE, PTYPE,
  PCODE, P and PERIOD (numeric, explicit).
  Required=Yes

### Fields:

### Parameters:

* **period**:
  Numeric integer period number for to be assigned to the input perimeters.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **tol**:
  Points in the perimeter file less than TOL distance apart will be deemed duplicate and
  rejected.(0.1)
  Range=Undefined
  Values=Undefined
  Default=0.1
  Required=No

* **ptype**:
  PTYPE field value [numeric] to be stored in the string file, representing a string
  type, if not found in perimeter file.(1)
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **pcode**:
  PCODE field value to be stored in string file, representing a line code 1001-1008, if
  not found in the perimeter file.(1001)
  Range=Undefined
  Values=Undefined
  Default=1001
  Required=No

* **p**:
  P field value [numeric] to be stored in point and string files, representing a point
  symbol and string line colour, if not found in perimeter file.(1)
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **psymbol**:
  PSYMBOL field value [numeric] to be stored in the point file, representing a point
  symbol number 91-98, if not found in the perimeter file.(92) Point symbol number 91 :
  Circle (o) 92 : Cross (+) 93 : Cross (x) 94 : Triangle 95 : Box 96 : Diamond 97 : Star (
  )
  Range=91,98
  Values=91,92,93,94,95,96,97,98
  Default=92
  Required=No

* **psymsze**:
  PSYMSZE field value [numeric] to be stored in the point file, representing a point
  symbol size in mm, if not found in the perimeter file.(1.5)
  Range=Undefined
  Values=Undefined
  Default=1.5
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('survin')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute survin
print("Running survin...")
dm_cmd.survin(
    perimin_i='t_input_file',  # required
    pointou_o='t_survin_out',  # required
    segou_o='t_survin_out',  # required
    period_p='required_val',  # required
    # tol_p=0.1,  # optional
    # ptype_p=1,  # optional
    # pcode_p=1001,  # optional
    # p_p=1,  # optional
    # psymbol_p=92,  # optional
    # psymsze_p=1.5,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("survin execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_survin_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")